# CiscoNet Expert — Asistente RAG para redes Cisco

Este proyecto implementa un asistente experto en redes Cisco utilizando Gemini, ChromaDB, LangGraph y RAG.

El objetivo es responder preguntas técnicas sobre conceptos y configuraciones CCNA, incluyendo VLANs, switching, routing, OSPF, ACLs, DHCP, NAT y troubleshooting.

In [1]:
# Instalación de dependencias principales
# Ejecutar solo si no están instaladas

%pip install -q langchain langchain-google-genai langgraph chromadb pypdf python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 1. Importación de librerías

En esta sección importamos las librerías necesarias para cargar documentos PDF, dividirlos en fragmentos, crear embeddings con Gemini, almacenar la información en ChromaDB y construir el agente con LangGraph.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_chroma import Chroma

from typing import TypedDict, List
from langgraph.graph import StateGraph, END

print("Librerías importadas correctamente")

Librerías importadas correctamente


c:\cisconet-expert-rag\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
%pip install -q langchain-community langchain-text-splitters langchain-chroma

Note: you may need to restart the kernel to use updated packages.


## 2. Configuración de la API Key

La clave de Gemini se carga desde un archivo `.env` para evitar escribir credenciales directamente en el notebook.

In [4]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if GOOGLE_API_KEY:
    print("API Key cargada correctamente")
else:
    print("No se ha encontrado GOOGLE_API_KEY. Revisa el archivo .env")

API Key cargada correctamente


## 3. Comprobación de documentos PDF

Antes de crear embeddings, comprobamos que los documentos están correctamente ubicados en la carpeta `docs/`.

In [5]:
docs_path = Path("docs")

pdf_files = list(docs_path.glob("*.pdf"))

print(f"PDFs encontrados: {len(pdf_files)}")

for pdf in pdf_files:
    print("-", pdf.name)

PDFs encontrados: 3
- 1. CCNA 200-301 Official Cert Guide, Volume 1 .pdf
- 2. CCNA 200-301 Official Cert Guide, Volume 2 .pdf
- CCNA Cisco Warrior .pdf


## 4. Carga inicial de PDFs

Cargamos las páginas de los documentos para verificar que el texto se extrae correctamente.

In [6]:
documents = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(str(pdf_file))
    pdf_documents = loader.load()
    
    for doc in pdf_documents:
        doc.metadata["source_file"] = pdf_file.name
    
    documents.extend(pdf_documents)

print(f"Total de páginas cargadas: {len(documents)}")
print("Primer documento cargado:")
print(documents[0].metadata)
print(documents[0].page_content[:1000])

Total de páginas cargadas: 2060
Primer documento cargado:
{'producer': '', 'creator': 'Adobe InDesign 14.0 (Macintosh)', 'creationdate': 'D:20190725145001', 'moddate': '2019-09-30T18:46:57+03:30', 'trapped': '/False', 'title': 'CCNA 200-301: Official Cert Guide, Volume 1', 'technet24': '{EF2A6058-968B-458F-821B-DADDFB529C6A}', 'codemantra, llc': 'http://www.codemantra.com', 'text watermark technet24': '{5A13A276-006B-4AE8-9DDD-2163AA19E3FE}', 'keywords': 'Technet24', 'author': 'Wendell Odom', 'subject': '', 'universal pdf': 'The process that creates this PDF constitutes a trade secret of codeMantra, LLC and is protected by the copyright laws of the United States', 't2': '{888DF979-9094-4875-8C35-325E6DAD0A59}', 'source': 'docs\\1. CCNA 200-301 Official Cert Guide, Volume 1 .pdf', 'total_pages': 1095, 'page': 0, 'page_label': 'Cover', 'source_file': '1. CCNA 200-301 Official Cert Guide, Volume 1 .pdf'}
ptg29743230
||||||||||||||||||||
||||||||||||||||||||


## 5. División de documentos (Chunking)

Los documentos se dividen en fragmentos más pequeños para mejorar la recuperación semántica en el sistema RAG.

Se utiliza `RecursiveCharacterTextSplitter` para generar chunks con solapamiento, permitiendo conservar contexto entre fragmentos.

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Total de chunks generados: {len(chunks)}")

Total de chunks generados: 6026


In [8]:
print(chunks[0].page_content)

ptg29743230
||||||||||||||||||||
||||||||||||||||||||


In [9]:
print(chunks[10].page_content)

service efforts designing and troubleshooting internetworks, performing data center and 
network audits, and assisting clients with their short- and long-term design objectives. 
Elan has a global perspective of network architectures via his international clientele. 
Elan has used his expertise to design and troubleshoot data centers and internetworks in 
Malaysia, North America, Europe, Australia, Africa, China, and the Middle East. Most 
recently, Elan has been focused on data center design, configuration, and troubleshoot-
ing as well as service provider technologies. In 1993, Elan was among the first to obtain 
the Cisco Certified System Instructor (CCSI) certification, and in 1996, he was among 
the first to attain the Cisco System highest technical certification, the Cisco Certified 
Internetworking Expert. Since then, Elan has been involved in numerous large-scale data 
center and telecommunications networking projects worldwide.
v
||||||||||||||||||||
||||||||||||||||||||


In [10]:
for chunk in chunks:
    chunk.metadata["chunk_size"] = len(chunk.page_content)

print(chunks[0].metadata)

{'producer': '', 'creator': 'Adobe InDesign 14.0 (Macintosh)', 'creationdate': 'D:20190725145001', 'moddate': '2019-09-30T18:46:57+03:30', 'trapped': '/False', 'title': 'CCNA 200-301: Official Cert Guide, Volume 1', 'technet24': '{EF2A6058-968B-458F-821B-DADDFB529C6A}', 'codemantra, llc': 'http://www.codemantra.com', 'text watermark technet24': '{5A13A276-006B-4AE8-9DDD-2163AA19E3FE}', 'keywords': 'Technet24', 'author': 'Wendell Odom', 'subject': '', 'universal pdf': 'The process that creates this PDF constitutes a trade secret of codeMantra, LLC and is protected by the copyright laws of the United States', 't2': '{888DF979-9094-4875-8C35-325E6DAD0A59}', 'source': 'docs\\1. CCNA 200-301 Official Cert Guide, Volume 1 .pdf', 'total_pages': 1095, 'page': 0, 'page_label': 'Cover', 'source_file': '1. CCNA 200-301 Official Cert Guide, Volume 1 .pdf', 'chunk_size': 53}


## 6. Limpieza de chunks

Se eliminan fragmentos vacíos o con contenido no legible para mejorar la calidad del sistema RAG.

In [11]:
clean_chunks = []

for chunk in chunks:
    
    text = chunk.page_content.strip()
    
    # Filtrar chunks demasiado pequeños
    if len(text) < 50:
        continue
    
    # Filtrar chunks con demasiados símbolos raros
    strange_ratio = text.count("|") / max(len(text), 1)
    
    if strange_ratio > 0.1:
        continue
    
    clean_chunks.append(chunk)

print(f"Chunks originales: {len(chunks)}")
print(f"Chunks limpios: {len(clean_chunks)}")

Chunks originales: 6026
Chunks limpios: 5698


In [12]:
print(clean_chunks[0].page_content)

ptg29743230
CCNA 200-301, Volume 1  
Official  Cert Guide
In addition to the wealth of updated content, this new edition includes a series of free 
hands-on exercises to help you master several real-world configuration and troubleshooting 
activities. These exercises can be performed on the CCNA 200-301 Network Simulator 
Lite, Volume 1 software included for free on the companion website that accompanies this 
book. This software, which simulates the experience of working on actual Cisco routers and 
switches, contains the following 21 free lab exercises, covering topics in Part II and Part III, 
the first hands-on configuration sections of the book:
 1.  Configuring Local Usernames
 2.  Configuring Hostnames
 3.  Interface Status I
 4.  Interface Status II
 5.  Interface Status III
 6.  Interface Status IV
 7.  Configuring Switch IP Settings
 8.  Switch IP Address
 9.  Switch IP Connectivity I
 10.  Switch CLI Configuration Process I
 11.  Switch CLI Configuration Process II


## 7. Creación de embeddings con Gemini

Los fragmentos limpios se transforman en embeddings vectoriales utilizando Gemini Embeddings.

Estos embeddings permiten realizar búsquedas semánticas sobre la documentación técnica Cisco.

## 8. Creación de la base vectorial ChromaDB

Los embeddings se almacenan en ChromaDB para permitir recuperación semántica eficiente dentro del sistema RAG.

In [13]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Modelo de embeddings cargado correctamente")

Modelo de embeddings cargado correctamente


In [14]:
test_embedding = embeddings.embed_query("What is a VLAN in Cisco networks?")

print(len(test_embedding))
print(test_embedding[:5])

3072
[-0.045847263, -0.032301076, -0.014878963, -0.040060762, -0.035875686]


In [15]:
vectorstore = Chroma.from_documents(
    documents=clean_chunks,
    embedding=embeddings,
    persist_directory="data/chromadb_cisco"
)

print("Base vectorial creada correctamente")

Base vectorial creada correctamente


## 9. Recuperación semántica (Retriever)

El retriever permite buscar los fragmentos más relevantes dentro de la base vectorial utilizando similitud semántica.

In [16]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Retriever creado correctamente")

Retriever creado correctamente


In [17]:
query = "How do you configure VLANs on a Cisco switch?"

results = retriever.invoke(query)

print(f"Resultados obtenidos: {len(results)}")

Resultados obtenidos: 4


In [18]:
for i, doc in enumerate(results):
    
    print("\n" + "="*80)
    print(f"RESULTADO {i+1}")
    print("="*80)
    
    print(doc.page_content[:1500])


RESULTADO 1
This chapter separates the VLAN configuration details into two major sections. The first sec-
tion looks at how to configure static access interfaces: switch interfaces configured to be in 
one VLAN only , therefore not using VLAN trunking. The second part shows how to config -
ure interfaces that do use VLAN trunking.
Creating VLANs and Assigning Access VLANs to an Interface
This section shows how to create a VLAN, give the VLAN a name, and assign interfaces to a 
VLAN. T o focus on these basic details, this section shows examples using a single switch, so 
VLAN trunking is not needed.
For a Cisco switch to forward frames in a particular VLAN, the switch must be configured 
to believe that the VLAN exists. In addition, the switch must have nontrunking interfaces 
(called access interfaces, or static access interfaces) assigned to the VLAN, and/or trunks 
that support the VLAN. The configuration steps for access interfaces are as follows:
Step

RESULTADO 2
This chapter sep

## 10. Inicialización del modelo Gemini

Se utiliza Gemini como modelo generativo principal para responder preguntas técnicas utilizando el contexto recuperado desde ChromaDB.

In [19]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

print("Modelo Gemini cargado correctamente")

Modelo Gemini cargado correctamente


## 11. Prompt Engineering

Se define un system prompt especializado en soporte y troubleshooting Cisco para controlar el comportamiento del asistente.

In [20]:
SYSTEM_PROMPT = """
You are CiscoNet Expert, an AI assistant specialized in Cisco networking.

Your responsibilities include:
- VLAN configuration
- Switching
- Routing
- OSPF
- ACLs
- NAT
- DHCP
- Cisco IOS troubleshooting

Rules:
1. Answer using the retrieved context.
2. If the answer is not present in the context, say so clearly.
3. Be technically accurate.
4. Provide step-by-step explanations.
5. Include Cisco IOS commands when appropriate.
6. Keep answers clear and professional.
"""

In [21]:
def generate_rag_response(question):

    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
    {SYSTEM_PROMPT}

    Context:
    {context}

    User Question:
    {question}

    Answer:
    """

    response = llm.invoke(prompt)

    return response.content

In [22]:
question = "How do you configure a VLAN on a Cisco switch?"

answer = generate_rag_response(question)

print(answer)

To configure a VLAN on a Cisco switch, you need to perform the following steps:

**Step 1: Create the VLAN and (optionally) name it.**

1.  From global configuration mode, use the `vlan vlan-id` command to create the VLAN and enter VLAN configuration mode.
    *   **Cisco IOS Command:** `vlan [vlan-id]`
    *   **Example:**
        ```cisco
        SW1# configure terminal
        SW1(config)# vlan 2
        ```
2.  (Optional) In VLAN configuration mode, use the `name name` command to assign a descriptive name to the VLAN. If not configured, the VLAN name will default to `VLAN ZZZZ`, where ZZZZ is the four-digit VLAN ID.
    *   **Cisco IOS Command:** `name [name]`
    *   **Example:**
        ```cisco
        SW1(config-vlan)# name Freds-vlan
        SW1(config-vlan)# exit
        ```

**Step 2: Assign interfaces to the newly created VLAN.**

1.  For each desired access interface, use the `interface type number` command in global configuration mode to enter interface configuration mode

## 12. Memoria conversacional

Se implementa memoria básica para mantener el contexto entre preguntas consecutivas dentro de la conversación.

In [23]:
conversation_history = ""

In [24]:
def generate_rag_response(question):

    global conversation_history

    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
    {SYSTEM_PROMPT}

    Conversation History:
    {conversation_history}

    Context:
    {context}

    User Question:
    {question}

    Answer:
    """

    response = llm.invoke(prompt)

    conversation_history += f"\nUser: {question}"
    conversation_history += f"\nAssistant: {response.content}\n"

    return response.content

In [25]:
question1 = "I have VLAN 10 and VLAN 20 configured."

answer1 = generate_rag_response(question1)

print(answer1)

Yes, based on the provided context, VLAN 10 and VLAN 20 are configured.

Here's a breakdown of their configuration:

**1. VLAN Creation:**
The context explicitly states the creation of VLAN 10 and VLAN 20.
```
vlan 10
name enet
vlan 20
name enet
```

**2. Port Assignments:**
Specific switch ports have been assigned to these VLANs in access mode:
*   **VLAN 10:**
    *   Interface FastEthernet0/1
    *   Interface FastEthernet0/2
    *   (The context also mentions PC1, PC2, PC5, and PC6 are in VLAN 10, implying more ports or inter-switch connectivity for PC5/PC6)
    ```
    Switch(config)#interface FastEthernet0/1
    Switch(config-if)#switchport mode access
    Switch(config-if)#switchport access vlan 10
    Switch(config-if)#exit
    Switch(config)#interface FastEthernet0/2
    Switch(config-if)#switchport mode access
    Switch(config-if)#switchport access vlan 10
    Switch(config-if)#exit
    ```
*   **VLAN 20:**
    *   Interface FastEthernet0/3
    *   Interface FastEthernet0/4


In [26]:
question2 = "How can I block VLAN 20 from accessing VLAN 10?"

answer2 = generate_rag_response(question2)

print(answer2)

Based on the provided context, the user is experiencing an inability to ping between different VLANs ("PC dapat melakukan ping ke sesame VLAN beda switch namun tidak bisa ke beda VLAN"). The context also explicitly states, "Untuk menghubungkan VLAN yang berbeda, dibutuhkan perangkat layer 3 baik itu router atau switch layer 3." This indicates that inter-VLAN routing is either not configured or not functioning, which is a prerequisite for blocking traffic between VLANs using an Access Control List (ACL).

To block VLAN 20 from accessing VLAN 10, you would typically use an Access Control List (ACL) on the Layer 3 device (router or Layer 3 switch) that performs the inter-VLAN routing.

**Step-by-step explanation to block VLAN 20 from accessing VLAN 10 (assuming inter-VLAN routing is configured on a Layer 3 device):**

1.  **Ensure Inter-VLAN Routing is Configured:**
    First, you need a Layer 3 device (a router or a Layer 3 switch with SVIs) to route traffic between VLAN 10 and VLAN 20. 

## 13. Implementación del agente con LangGraph

Se implementa un flujo basado en LangGraph para estructurar el proceso RAG en dos etapas:

1. Recuperación semántica de contexto.
2. Generación de respuesta mediante Gemini.

Este enfoque permite construir agentes modulares y escalables.

In [27]:
class GraphState(TypedDict):
    
    question: str
    context: str
    answer: str
    history: str

In [28]:
def retrieve_node(state):

    question = state["question"]

    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    return {
        "context": context
    }

In [29]:
from tenacity import retry, stop_after_attempt, wait_exponential

In [30]:
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=2, min=2, max=12)
)
def safe_llm_invoke(prompt):
    
    return llm.invoke(prompt)

In [31]:
def generate_node(state):

    prompt = f"""
    {SYSTEM_PROMPT}

    Conversation History:
    {state['history']}

    Context:
    {state['context']}

    User Question:
    {state['question']}

    Answer:
    """

    response = safe_llm_invoke(prompt)

    updated_history = (
        state["history"]
        + f"\nUser: {state['question']}"
        + f"\nAssistant: {response.content}\n"
    )

    return {
        "answer": response.content,
        "history": updated_history
    }

In [32]:
workflow = StateGraph(GraphState)

workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)

workflow.set_entry_point("retrieve")

workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()

print("LangGraph compilado correctamente")

LangGraph compilado correctamente


In [33]:
conversation_history = ""

def chat(question):

    global conversation_history

    result = app.invoke({
        "question": question,
        "history": conversation_history
    })

    conversation_history = result["history"]

    return result["answer"]

In [34]:
response1 = chat(
    "How do I configure OSPF on a Cisco router?"
)

print(response1)

To configure OSPF on a Cisco router, you need to enable the OSPF routing process and then define the networks that will participate in OSPF, assigning them to specific areas.

Here are the step-by-step instructions with Cisco IOS commands:

1.  **Enter Global Configuration Mode:**
    First, access the global configuration mode on your router.
    ```cisco
    SEMARANG#configure terminal
    SEMARANG(config)#
    ```

2.  **Enable OSPF Process and Define Process ID:**
    Use the `router ospf` command followed by a process ID. The process ID is a locally significant number (1-65535) that identifies the OSPF routing process on the router.
    ```cisco
    SEMARANG(config)#router ospf 1
    SEMARANG(config-router)#
    ```
    *Example from context: `router ospf 1` for SEMARANG and `router ospf 2` for SOLO.*

3.  **Define Networks and Area Numbers:**
    Use the `network` command to specify which interfaces will participate in OSPF and to which OSPF area they belong. This command require

In [35]:
response2 = chat(
    "How can I restrict access between VLANs using ACLs?"
)

print(response2)

The provided context discusses VLAN configuration and mentions the general use of ACLs to match and filter traffic types like Telnet, SSH, ping, and traceroute. It also provides examples of assigning interfaces to VLANs (e.g., `switchport access vlan 10`, `switchport access vlan 20`).

However, the context does not contain specific instructions or Cisco IOS commands on *how to apply ACLs to restrict access between VLANs* at the Layer 3 routing point (e.g., on a router's subinterfaces or a Layer 3 switch's SVI interfaces). It only indicates that ACLs can be created to deny or permit certain traffic.


## 15. Chat interactivo (opcional)

El sistema permite conversación interactiva mediante un bucle continuo de preguntas y respuestas.

Código utilizado:

```python
while True:

    question = input("\nPregunta: ")

    if question.lower() == "salir":
        break

    response = chat(question)

    print("\nCiscoNet Expert:\n")
    print(response)

## 15. Ejemplos de conversación documentados

A continuación se muestran cinco consultas de prueba para validar el funcionamiento del asistente CiscoNet Expert.

Los ejemplos cubren distintos dominios técnicos del temario CCNA: VLANs, ACLs, OSPF, NAT y troubleshooting.

In [36]:
example_1 = "How do I configure a VLAN on a Cisco switch and assign an access port to it?"

response_1 = chat(example_1)

print("EXAMPLE 1 - VLAN Configuration")
print("="*80)
print(response_1)

EXAMPLE 1 - VLAN Configuration
To configure a VLAN on a Cisco switch and assign an access port to it, follow these steps:

1.  **Enter Global Configuration Mode:**
    First, access the global configuration mode on your switch.
    ```cisco
    Switch#configure terminal
    Switch(config)#
    ```

2.  **Configure a New VLAN:**
    A.  Use the `vlan` command followed by the desired VLAN ID in global configuration mode to create the VLAN. This will also move you into VLAN configuration mode.
        ```cisco
        Switch(config)#vlan 10
        Switch(config-vlan)#
        ```
    B.  (Optional) Use the `name` command to assign a descriptive name to the VLAN. If not configured, the VLAN name will default to `VLAN ZZZZ`, where `ZZZZ` is the four-digit VLAN ID.
        ```cisco
        Switch(config-vlan)#name Sales
        Switch(config-vlan)#exit
        Switch(config)#
        ```

3.  **Assign an Access Port to the VLAN:**
    A.  Enter interface configuration mode for the specific 

In [37]:
example_2 = "I have VLAN 10 and VLAN 20. How can I block VLAN 20 from accessing VLAN 10 using ACLs?"

response_2 = chat(example_2)

print("EXAMPLE 2 - ACL Between VLANs")
print("="*80)
print(response_2)

EXAMPLE 2 - ACL Between VLANs
The provided context discusses VLAN configuration, assigning ports to VLANs, creating Switched Virtual Interfaces (SVIs) with IP addresses, and enabling IP routing for inter-VLAN communication.

However, the context does not contain specific instructions or Cisco IOS commands on *how to configure and apply Access Control Lists (ACLs) to block traffic between VLANs* (e.g., blocking VLAN 20 from accessing VLAN 10). It only shows the setup for inter-VLAN routing to occur.


In [38]:
example_3 = "How do I configure single-area OSPF on Cisco routers and verify neighbor adjacency?"

response_3 = chat(example_3)

print("EXAMPLE 3 - OSPF Configuration")
print("="*80)
print(response_3)

EXAMPLE 3 - OSPF Configuration
To configure single-area OSPF on Cisco routers and verify neighbor adjacency, follow these steps:

### 1. Configure Single-Area OSPF

To configure single-area OSPF, you will enable the OSPF routing process and define the networks that will participate in OSPF, assigning them all to the same area (e.g., Area 0, the backbone area).

Here are the step-by-step instructions with Cisco IOS commands:

1.  **Enter Global Configuration Mode:**
    ```cisco
    Router#configure terminal
    Router(config)#
    ```

2.  **Enable OSPF Process and Define Process ID:**
    Use the `router ospf` command followed by a process ID. This process ID is locally significant.
    ```cisco
    Router(config)#router ospf 1
    Router(config-router)#
    ```

3.  **Define Networks and Assign to a Single Area (e.g., Area 0):**
    Use the `network` command to specify which interfaces will participate in OSPF and to which OSPF area they belong. For single-area OSPF, all participatin

In [39]:
example_4 = "How do I configure NAT overload or PAT on a Cisco router for internal users accessing the Internet?"

response_4 = chat(example_4)

print("EXAMPLE 4 - NAT Overload / PAT")
print("="*80)
print(response_4)

EXAMPLE 4 - NAT Overload / PAT
To configure NAT Overload (PAT) on a Cisco router for internal users accessing the Internet, you need to identify the inside and outside interfaces, define an Access Control List (ACL) to match the internal traffic, and then apply the NAT overload command. This configuration allows multiple internal private IP addresses to share a single public IP address for external communication by using unique port numbers.

Here are the step-by-step instructions with Cisco IOS commands, based on the provided context:

1.  **Enter Global Configuration Mode:**
    ```cisco
    Router#configure terminal
    Router(config)#
    ```

2.  **Identify Inside Interfaces:**
    Configure the interface(s) connected to your internal network as `ip nat inside`.
    ```cisco
    Router(config)#interface GigabitEthernet0/0  (Example: Replace with your actual inside interface)
    Router(config-if)#ip nat inside
    Router(config-if)#exit
    Router(config)#
    ```

3.  **Identify 

In [40]:
example_5 = "A router cannot ping a remote network. What troubleshooting steps should I follow using Cisco IOS commands?"

response_5 = chat(example_5)

print("EXAMPLE 5 - IPv4 Routing Troubleshooting")
print("="*80)
print(response_5)

EXAMPLE 5 - IPv4 Routing Troubleshooting
When a Cisco router cannot ping a remote network, you should follow a systematic troubleshooting approach using Cisco IOS commands to identify the root cause. The `ping` command uses ICMP to test connectivity by sending packets and expecting replies, helping to isolate problems.

Here are the troubleshooting steps with Cisco IOS commands:

1.  **Verify Local Interface Status and IP Configuration:**
    The context mentions "IP address configuration problems" and "Router interface problems (shutdown, wrong IP)" as potential causes for ping failures. Ensure that the router's interface connected towards the remote network (or its next-hop router) is administratively up, operationally up, and has the correct IP address configured.

    *   **Cisco IOS Commands:**
        ```cisco
        Router#show ip interface brief
        ```
        This command provides a summary of all interfaces, their IP addresses, and their status (up/down, administrativel

## 16. Prueba específica de memoria conversacional

Este ejemplo demuestra que el agente mantiene contexto entre turnos.

In [41]:
conversation_history = ""

memory_q1 = "I am working with VLAN 10 using subnet 10.10.10.0/24 and VLAN 20 using subnet 20.20.20.0/24."

memory_a1 = chat(memory_q1)

print(memory_a1)

Based on the provided context, you are indeed working with:

*   **VLAN 10** using the subnet **10.10.10.0/24**.
    *   Interfaces `f0/1` and `f0/2` are configured as access ports for VLAN 10.
    *   Devices with IP addresses like `10.10.10.10/24` and `10.10.10.11/24` are associated with this VLAN.

*   **VLAN 20** using the subnet **20.20.20.0/24**.
    *   Interfaces `f0/3` and `f0/4` are configured as access ports for VLAN 20.
    *   Devices with IP addresses like `20.20.20.20/24` and `20.20.20.21/24` are associated with this VLAN.


In [42]:
memory_q2 = "Now block the second VLAN from accessing the first one."

memory_a2 = chat(memory_q2)

print(memory_a2)

Based on the provided context, VLAN 20 is already unable to access VLAN 10. The output `PC tidak bisa ping ke beda VLAN` (PC cannot ping to different VLANs) and the failed ping from a PC in VLAN 10 (10.10.10.x) to a PC in VLAN 20 (20.20.20.21) confirm this.

**Explanation:**

1.  **VLAN Isolation:** VLANs are designed to segment a network into separate broadcast domains. By default, devices in one VLAN cannot communicate directly with devices in another VLAN at Layer 3 (IP layer) without a routing device.
2.  **Lack of Inter-VLAN Routing:** The provided context does not show any configuration for a Layer 3 device (such as a router or a Layer 3 switch with Switched Virtual Interfaces - SVIs) that would enable inter-VLAN routing between VLAN 10 and VLAN 20. Therefore, the two VLANs are currently isolated, and VLAN 20 cannot access VLAN 10.

**If Inter-VLAN Routing Were Configured (Hypothetical Scenario):**

If inter-VLAN routing were enabled, allowing VLAN 20 to communicate with VLAN 10,

# 17. Conclusiones

CiscoNet Expert demuestra la implementación de un sistema RAG especializado en redes Cisco utilizando Gemini, ChromaDB y LangGraph.

El sistema permite:
- Recuperación semántica de documentación técnica.
- Generación de respuestas contextualizadas mediante Gemini.
- Memoria conversacional básica.
- Resolución de consultas técnicas sobre VLANs, ACLs, OSPF, NAT y troubleshooting Cisco IOS.

Tecnologías utilizadas:
- Gemini 2.5 Flash
- Gemini Embeddings
- ChromaDB
- LangChain
- LangGraph
- Python

La arquitectura implementada permite construir asistentes especializados capaces de consultar documentación técnica y generar respuestas precisas utilizando Retrieval-Augmented Generation (RAG).